In [72]:
import argparse
import yaml
import duckdb
import glob
import os
import re


In [73]:
config_path = '/n/dominici_nsaph_l3/Lab/data_processing/shreya_synthetic-cms/synthetic-cms/utils/medicare.yml'

In [74]:
# read in yaml containing harmonization rules
with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

In [68]:
# build input parquet path based upon pattern and basepath in yaml
def get_parquet_files(basepath, year, path_pattern):
    year_path = os.path.join(basepath, str(year))
    # add year to path
    path_pattern = path_pattern.replace("{basepath}", basepath).replace("{year}", str(year))
    # extract pattern of filetype 
    dir_pattern = os.path.dirname(path_pattern)
    # find directories that match the directory pattern
    matched_dirs = [d for d in glob.glob(dir_pattern) if os.path.isdir(d)]
    
    # list full paths of each chunk
    parquet_files = []
    for directory in matched_dirs:
        parquet_files.extend(sorted(glob.glob(os.path.join(directory, "part-*.parquet"))))
    
    return parquet_files


In [69]:
parquet_files = get_parquet_files(basepath,2016,'{basepath}/{year}/mbsf_*d*/part-*.parquet')
print(parquet_files)

['/n/dominici_nsaph_l3/Lab/data/data_warehouse/dw_raw_medicare/2016/mbsf_abcd_summary_res000017155_req008183/part-01.parquet', '/n/dominici_nsaph_l3/Lab/data/data_warehouse/dw_raw_medicare/2016/mbsf_abcd_summary_res000017155_req008183/part-02.parquet', '/n/dominici_nsaph_l3/Lab/data/data_warehouse/dw_raw_medicare/2016/mbsf_abcd_summary_res000017155_req008183/part-03.parquet', '/n/dominici_nsaph_l3/Lab/data/data_warehouse/dw_raw_medicare/2016/mbsf_abcd_summary_res000017155_req008183/part-04.parquet', '/n/dominici_nsaph_l3/Lab/data/data_warehouse/dw_raw_medicare/2016/mbsf_abcd_summary_res000017155_req008183/part-05.parquet', '/n/dominici_nsaph_l3/Lab/data/data_warehouse/dw_raw_medicare/2016/mbsf_abcd_summary_res000017155_req008183/part-06.parquet']


In [70]:
# build a separate query for each table in the yaml 
def construct_query(table_config, parquet_files):
    columns = []
    
    for col in table_config.get("columns", []):
        if isinstance(col, dict):
            col_name = list(col.keys())[0]
            col_def = col[col_name]
            
            if "m" in col_def and "source" in col_def:
                # Handle multi-source case (e.g., for dual_indicators)
                expanded_sources = [col_def["source"][0].replace("{m}", m) for m in col_def["m"]]
                cast_expr = col_def.get("cast", {}).get("*", "array_value({columns})").format(columns=", ".join(expanded_sources))
                columns.append(f"{cast_expr} AS {col_name}")
            else:
                # Handle regular column case (e.g., for dual_mo)
                source = col_def.get("rename_source", col_name)
                cast = col_def.get("cast", {}).get("*", "{column_name}")
                columns.append(f"{cast.format(column_name=source)} AS {col_name}")

        else:
            # Handle case for direct column names
            columns.append(col)

    columns_str = ", ".join(columns)
    files_str = ", ".join([f"'{file}'" for file in parquet_files])

    return f"""
        CREATE OR REPLACE TABLE {table_config['name']} AS
        SELECT {columns_str}
        FROM read_parquet([{files_str}]);
    """



In [71]:
construct_query(config['tables']['mbsf_d'],parquet_files)
construct_query(config['tables']['ps'],parquet_files)

"\n        CREATE OR REPLACE TABLE ps AS\n        SELECT bene_id, file, record\n        FROM read_parquet(['/n/dominici_nsaph_l3/Lab/data/data_warehouse/dw_raw_medicare/2016/mbsf_abcd_summary_res000017155_req008183/part-01.parquet', '/n/dominici_nsaph_l3/Lab/data/data_warehouse/dw_raw_medicare/2016/mbsf_abcd_summary_res000017155_req008183/part-02.parquet', '/n/dominici_nsaph_l3/Lab/data/data_warehouse/dw_raw_medicare/2016/mbsf_abcd_summary_res000017155_req008183/part-03.parquet', '/n/dominici_nsaph_l3/Lab/data/data_warehouse/dw_raw_medicare/2016/mbsf_abcd_summary_res000017155_req008183/part-04.parquet', '/n/dominici_nsaph_l3/Lab/data/data_warehouse/dw_raw_medicare/2016/mbsf_abcd_summary_res000017155_req008183/part-05.parquet', '/n/dominici_nsaph_l3/Lab/data/data_warehouse/dw_raw_medicare/2016/mbsf_abcd_summary_res000017155_req008183/part-06.parquet']);\n    "

In [54]:
def process_tables(config, output_path):
    conn = duckdb.connect(database=':memory:')
    # retrieve basepath for input files from yaml
    basepath = config['basepath']
    # create output directory
    os.makedirs(output_path, exist_ok=True)
    # extract years from basepath
    years = [d for d in os.listdir(basepath) if d.isdigit()]
    years = sorted(map(int, years))
    # execute necessary harmonization for each table in yaml
    for table_name, table_config in config['tables'].items():
        table_config['name'] = table_name
        for year in years:
            parquet_files = get_parquet_files(basepath, year, table_config['path_pattern'])
            if parquet_files:
                query = construct_query(table_config, parquet_files)
                conn.execute(query)
                output_file = os.path.join(output_path, f"{table_name}_{year}.parquet")
                conn.execute(f"COPY (SELECT * FROM {table_name}) TO '{output_file}' (FORMAT 'parquet')")
                print(f"Processed {table_name} for {year} and saved to {output_file}")
    
    conn.close()

In [55]:
process_tables(config,"/n/dominici_nsaph_l3/Lab/data_processing/shreya_synthetic-cms/synthetic-cms/output")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Processed mbsf_d for 2011 and saved to /n/dominici_nsaph_l3/Lab/data_processing/shreya_synthetic-cms/synthetic-cms/output/mbsf_d_2011.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Error: KeyboardInterrupt: <EMPTY MESSAGE>

At:
  /n/helmod/apps/centos7/Core/Anaconda3/2021.05-jupyterood-fasrc01/x/lib/python3.8/site-packages/traitlets/traitlets.py(595): __set__
  <ipython-input-54-50363c771f81>(15): process_tables
  <ipython-input-55-6c14249cf7ed>(1): <module>
  /n/helmod/apps/centos7/Core/Anaconda3/2021.05-jupyterood-fasrc01/x/lib/python3.8/site-packages/IPython/core/interactiveshell.py(3437): run_code
  /n/helmod/apps/centos7/Core/Anaconda3/2021.05-jupyterood-fasrc01/x/lib/python3.8/site-packages/IPython/core/interactiveshell.py(3357): run_ast_nodes
  /n/helmod/apps/centos7/Core/Anaconda3/2021.05-jupyterood-fasrc01/x/lib/python3.8/site-packages/IPython/core/interactiveshell.py(3165): run_cell_async
  /n/helmod/apps/centos7/Core/Anaconda3/2021.05-jupyterood-fasrc01/x/lib/python3.8/site-packages/IPython/core/async_helpers.py(68): _pseudo_sync_runner
  /n/helmod/apps/centos7/Core/Anaconda3/2021.05-jupyterood-fasrc01/x/lib/python3.8/site-packages/IPython/core/interac

In [56]:
import pandas as pd
import pyarrow.parquet as pq
import duckdb

# Path to the Parquet file
file_path = "/n/dominici_nsaph_l3/Lab/data_processing/shreya_synthetic-cms/synthetic-cms/output/mbsf_d_2011.parquet"

# query = f" CREATE OR REPLACE TABLE mbsf_d AS SELECT bene_id, RFRNC_YR AS year, dual_mo::INT AS dual_mo, array_value(DUAL_01, DUAL_02, DUAL_03, DUAL_04, DUAL_05, DUAL_06, DUAL_07, DUAL_08, DUAL_09, DUAL_10, DUAL_11, DUAL_12) AS dual_indicators, 
# FROM read_parquet('/n/dominici_nsaph_l3/Lab/data/data_warehouse/dw_raw_medicare/2016/mbsf_abcd_summary_res000017155_req008183/part-01.parquet'),
# GROUP BY bene_id, year, dual_mo;"

# Create a DuckDB connection and query the first 10 rows
query = f"SELECT * FROM read_parquet('{file_path}') LIMIT 10"

# Execute the query and fetch the results into a Pandas DataFrame
df = duckdb.query(query).df()

# Show the first 10 rows
print(df)


           BENE_ID    year  dual_mo
0  llllllllllllllS  2011.0       12
1  lllllllllllll0l  2011.0        0
2  lllllllllllll07  2011.0        0
3  lllllllllllll0S  2011.0        0
4  lllllllllllll08  2011.0        0
5  lllllllllllllU0  2011.0       12
6  lllllllllllllU4  2011.0       12
7  lllllllllllll40  2011.0        0
8  lllllllllllll47  2011.0        0
9  lllllllllllllO0  2011.0       12
